# Silver Layer — ERP Customer (CUST_AZ12)
Clean and normalize `erp_cust_az12`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "silver",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## Read Bronze Table

In [ ]:
df = session.table(f"bronze.erp_cust_az12")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Customer ID Cleanup
Remove "NAS" prefix if present.

In [ ]:
df = df.with_column(
    "cid",
    F.when(F.col("cid").startswith("NAS"),
           F.substring(F.col("cid"), 4, F.length(F.col("cid"))))
     .otherwise(F.col("cid"))
)

### Birthdate Validation
Null out future dates.

In [ ]:
df = df.with_column(
    "bdate",
    F.when(F.col("bdate") > F.current_date(), F.lit(None))
     .otherwise(F.col("bdate"))
)

### Gender Normalization

In [ ]:
df = df.with_column(
    "gen",
    F.when(F.upper(F.col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(F.col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

### Rename Columns

In [ ]:
RENAME_MAP = {
    "cid":   "customer_number",
    "bdate": "birth_date",
    "gen":   "gender",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"silver.erp_customers", mode="overwrite")
print("erp_customers OK")

## Verify

In [ ]:
session.table(f"silver.erp_customers").limit(5).show()